In [1]:
import sys
sys.path.append("..")
from functools import partial
import pandas as pd
from src.data import load_cuad, TARGET_CLAUSES
from src.split import split_contracts
from src.chunking import clause_chunks
from src.retrieval import make_dense_ranker
from src.bm25 import make_bm25_ranker
from src.hybrid import make_hybrid_ranker
from src.metrics import evaluate_ranker

contracts = load_cuad()
dev_idx, test_idx = split_contracts(contracts)

chunker = partial(clause_chunks, max_size=1500)
dense = make_dense_ranker(chunker, tag="clause-200-1500-200")
bm25 = make_bm25_ranker(chunker)

In [2]:
import hashlib
from src.embed import MODEL_NAME, CACHE_DIR

tag = "clause-200-1500-200"
missing = 0
for i in dev_idx:
    key = hashlib.md5(f"{MODEL_NAME}|{tag}|{contracts[i]['title']}".encode()).hexdigest()
    if not (CACHE_DIR / f"{key}.npy").exists():
        missing += 1
print("dev contracts missing from cache:", missing)

dev contracts missing from cache: 40


In [3]:
systems = {
    "dense": dense,
    "bm25": bm25,
    "hybrid_1_1": make_hybrid_ranker([dense, bm25]),
    "hybrid_1_2": make_hybrid_ranker([dense, bm25], weights=[1.0, 2.0]),
}

results = {}
for name, r in systems.items():
    rows = {cl: evaluate_ranker(r, contracts, dev_idx, cl) for cl in TARGET_CLAUSES}
    results[name] = pd.DataFrame(rows).T
    results[name].to_csv(f"../eval/results_{name}_clause1500_dev.csv")
    print("done:", name)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

done: dense
done: bm25
done: hybrid_1_1
done: hybrid_1_2


In [4]:
summary = pd.DataFrame(
    {k: v[["hit@1", "hit@3", "mrr"]].mean() for k, v in results.items()}
).T.round(3)
print(summary)
print()
print(pd.DataFrame({k: v["hit@1"] for k, v in results.items()}).round(3))

            hit@1  hit@3    mrr
dense       0.611  0.800  0.723
bm25        0.751  0.891  0.827
hybrid_1_1  0.719  0.899  0.817
hybrid_1_2  0.747  0.921  0.836

                                    dense   bm25  hybrid_1_1  hybrid_1_2
Governing Law                       0.683  0.948       0.837       0.918
Renewal Term                        0.822  0.890       0.881       0.881
Notice Period To Terminate Renewal  0.620  0.718       0.648       0.676
Termination For Convenience         0.449  0.788       0.602       0.712
Non-Compete                         0.240  0.507       0.440       0.427
Cap On Liability                    0.700  0.853       0.784       0.816
Uncapped Liability                  0.765  0.556       0.840       0.802
